# 02 - Dataset Cleaning for YOLOv8 Retail Theft Detection

**Purpose:** Clean and fix dataset issues identified during validation

**Objectives:**
- Remove unreadable or corrupted images
- Remove or correct invalid bounding boxes
- Detect and remove duplicated samples
- Normalize file naming conventions
- Ensure strict one-to-one image-label matching
- Auto-fix common issues where possible

---

## 1. Setup and Imports

In [1]:
# Install required packages if needed
!pip install opencv-python pillow numpy pandas tqdm imagehash --quiet

In [2]:
import os
import sys
import cv2
import json
import shutil
import hashlib
import logging
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Set

import numpy as np
import pandas as pd
from PIL import Image
import imagehash
from tqdm import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("All imports successful!")

All imports successful!


In [3]:
# Configuration
BASE_DIR = Path(r"c:/Users/shaho/OneDrive - Nile University/Desktop/AletrixGrad")
DATASET_DIR = BASE_DIR / "cc-tv-footage-annotation-b8-lcysc-b1-2"
CLEANED_DIR = BASE_DIR / "dataset_cleaned"
BACKUP_DIR = BASE_DIR / "dataset_backup"
OUTPUT_DIR = BASE_DIR / "outputs"
LOG_DIR = BASE_DIR / "logs"

# Create directories
for dir_path in [CLEANED_DIR, BACKUP_DIR, OUTPUT_DIR, LOG_DIR]:
    dir_path.mkdir(exist_ok=True)

# Class definitions
CLASS_NAMES = {
    0: 'Customer-Bagpack',
    1: 'Product',
    2: 'Product-Picked',
    3: 'Shopping-Cart',
    4: 'normal',
    5: 'theft'
}
NUM_CLASSES = 6

# Image extensions
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

print(f"Source Dataset: {DATASET_DIR}")
print(f"Cleaned Dataset: {CLEANED_DIR}")
print(f"Backup Directory: {BACKUP_DIR}")

Source Dataset: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\cc-tv-footage-annotation-b8-lcysc-b1-2
Cleaned Dataset: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_cleaned
Backup Directory: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_backup


## 2. Dataset Cleaner Class

In [4]:
class DatasetCleaner:
    """Comprehensive YOLO dataset cleaner."""
    
    def __init__(self, source_dir: Path, output_dir: Path, num_classes: int = 6):
        self.source_dir = Path(source_dir)
        self.output_dir = Path(output_dir)
        self.num_classes = num_classes
        
        self.stats = {
            'total_images': 0,
            'cleaned_images': 0,
            'removed_corrupt': 0,
            'removed_duplicates': 0,
            'fixed_labels': 0,
            'removed_orphan_labels': 0,
            'created_missing_labels': 0,
            'fixed_bbox': 0,
            'removed_invalid_bbox': 0
        }
        
        self.cleaning_log = []
        self.image_hashes = {}  # For duplicate detection
        
    def compute_image_hash(self, image_path: Path) -> Optional[str]:
        """Compute perceptual hash of an image for duplicate detection."""
        try:
            with Image.open(image_path) as img:
                # Use perceptual hash for near-duplicate detection
                phash = str(imagehash.phash(img))
                return phash
        except Exception as e:
            logger.warning(f"Could not hash {image_path}: {e}")
            return None
    
    def is_image_valid(self, image_path: Path) -> Tuple[bool, str]:
        """Check if an image is valid and readable."""
        try:
            # Check file exists and has content
            if not image_path.exists():
                return False, "File does not exist"
            if image_path.stat().st_size == 0:
                return False, "File is empty"
            
            # Try PIL
            with Image.open(image_path) as img:
                img.verify()
            
            # Try OpenCV
            cv_img = cv2.imread(str(image_path))
            if cv_img is None:
                return False, "OpenCV cannot read image"
            
            return True, "OK"
            
        except Exception as e:
            return False, str(e)
    
    def fix_label_file(self, label_path: Path, image_path: Path) -> Tuple[bool, List[str], int]:
        """Fix common issues in a YOLO label file.
        
        Returns:
            Tuple of (success, fixed_lines, num_fixes)
        """
        if not label_path.exists():
            return False, [], 0
        
        try:
            with open(label_path, 'r') as f:
                lines = f.readlines()
        except Exception as e:
            logger.error(f"Cannot read {label_path}: {e}")
            return False, [], 0
        
        fixed_lines = []
        num_fixes = 0
        
        for line_num, line in enumerate(lines, 1):
            line = line.strip()
            if not line:
                continue
            
            parts = line.split()
            if len(parts) != 5:
                self.cleaning_log.append({
                    'action': 'remove_malformed_line',
                    'file': str(label_path),
                    'line': line_num,
                    'reason': f'Expected 5 values, got {len(parts)}'
                })
                num_fixes += 1
                continue
            
            try:
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])
            except ValueError:
                self.cleaning_log.append({
                    'action': 'remove_invalid_values',
                    'file': str(label_path),
                    'line': line_num,
                    'reason': 'Non-numeric values'
                })
                num_fixes += 1
                continue
            
            # Fix class ID
            if class_id < 0 or class_id >= self.num_classes:
                self.cleaning_log.append({
                    'action': 'remove_invalid_class',
                    'file': str(label_path),
                    'line': line_num,
                    'reason': f'Invalid class ID: {class_id}'
                })
                num_fixes += 1
                continue
            
            # Fix bounding box coordinates
            fixed = False
            
            # Clamp values to [0, 1] range
            if x_center < 0 or x_center > 1:
                x_center = max(0, min(1, x_center))
                fixed = True
            if y_center < 0 or y_center > 1:
                y_center = max(0, min(1, y_center))
                fixed = True
            if width <= 0 or width > 1:
                if width <= 0:
                    # Skip invalid width
                    self.cleaning_log.append({
                        'action': 'remove_zero_width',
                        'file': str(label_path),
                        'line': line_num,
                        'reason': f'Zero or negative width: {width}'
                    })
                    num_fixes += 1
                    continue
                width = min(1, width)
                fixed = True
            if height <= 0 or height > 1:
                if height <= 0:
                    self.cleaning_log.append({
                        'action': 'remove_zero_height',
                        'file': str(label_path),
                        'line': line_num,
                        'reason': f'Zero or negative height: {height}'
                    })
                    num_fixes += 1
                    continue
                height = min(1, height)
                fixed = True
            
            # Ensure bbox doesn't extend beyond image
            x_min = x_center - width / 2
            x_max = x_center + width / 2
            y_min = y_center - height / 2
            y_max = y_center + height / 2
            
            if x_min < 0:
                width = width + x_min * 2
                x_center = width / 2
                fixed = True
            if x_max > 1:
                width = (1 - x_center) * 2
                fixed = True
            if y_min < 0:
                height = height + y_min * 2
                y_center = height / 2
                fixed = True
            if y_max > 1:
                height = (1 - y_center) * 2
                fixed = True
            
            # Skip very small bboxes (noise)
            min_area = 0.0001  # 0.01% of image
            if width * height < min_area:
                self.cleaning_log.append({
                    'action': 'remove_tiny_bbox',
                    'file': str(label_path),
                    'line': line_num,
                    'reason': f'Bbox too small: {width * height:.6f}'
                })
                num_fixes += 1
                continue
            
            if fixed:
                num_fixes += 1
                self.cleaning_log.append({
                    'action': 'fix_bbox_coordinates',
                    'file': str(label_path),
                    'line': line_num,
                    'reason': 'Clamped coordinates to valid range'
                })
            
            # Add fixed line
            fixed_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        
        return True, fixed_lines, num_fixes
    
    def find_duplicates(self, split_path: Path) -> Dict[str, List[Path]]:
        """Find duplicate images using perceptual hashing."""
        images_path = split_path / "images"
        if not images_path.exists():
            return {}
        
        hash_to_files = defaultdict(list)
        
        image_files = [f for f in images_path.iterdir() 
                       if f.suffix.lower() in IMAGE_EXTENSIONS]
        
        for img_path in tqdm(image_files, desc="Computing hashes"):
            img_hash = self.compute_image_hash(img_path)
            if img_hash:
                hash_to_files[img_hash].append(img_path)
        
        # Return only groups with duplicates
        duplicates = {h: files for h, files in hash_to_files.items() if len(files) > 1}
        return duplicates
    
    def clean_split(self, split_name: str) -> Dict:
        """Clean a single dataset split."""
        source_split = self.source_dir / split_name
        output_split = self.output_dir / split_name
        
        # Create output directories
        (output_split / "images").mkdir(parents=True, exist_ok=True)
        (output_split / "labels").mkdir(parents=True, exist_ok=True)
        
        source_images = source_split / "images"
        source_labels = source_split / "labels"
        
        if not source_images.exists():
            logger.warning(f"Source images not found: {source_images}")
            return {}
        
        split_stats = {
            'total': 0,
            'cleaned': 0,
            'removed_corrupt': 0,
            'removed_duplicates': 0,
            'fixed_labels': 0
        }
        
        # Find duplicates first
        logger.info(f"Finding duplicates in {split_name}...")
        duplicates = self.find_duplicates(source_split)
        
        # Track which files to skip (duplicates)
        skip_files = set()
        for hash_val, files in duplicates.items():
            # Keep the first file, skip the rest
            for dup_file in files[1:]:
                skip_files.add(dup_file.name)
                self.cleaning_log.append({
                    'action': 'remove_duplicate',
                    'file': str(dup_file),
                    'reason': f'Duplicate of {files[0].name}'
                })
        
        split_stats['removed_duplicates'] = len(skip_files)
        logger.info(f"Found {len(skip_files)} duplicates to remove")
        
        # Process images
        image_files = [f for f in source_images.iterdir() 
                       if f.suffix.lower() in IMAGE_EXTENSIONS]
        split_stats['total'] = len(image_files)
        
        logger.info(f"Processing {len(image_files)} images in {split_name}...")
        
        for img_path in tqdm(image_files, desc=f"Cleaning {split_name}"):
            # Skip duplicates
            if img_path.name in skip_files:
                continue
            
            # Validate image
            is_valid, reason = self.is_image_valid(img_path)
            if not is_valid:
                split_stats['removed_corrupt'] += 1
                self.cleaning_log.append({
                    'action': 'remove_corrupt',
                    'file': str(img_path),
                    'reason': reason
                })
                continue
            
            # Process label file
            label_path = source_labels / (img_path.stem + '.txt')
            
            if label_path.exists():
                success, fixed_lines, num_fixes = self.fix_label_file(label_path, img_path)
                
                if num_fixes > 0:
                    split_stats['fixed_labels'] += 1
            else:
                # Create empty label file if missing
                fixed_lines = []
                self.cleaning_log.append({
                    'action': 'create_empty_label',
                    'file': str(label_path),
                    'reason': 'Missing label file for image'
                })
                self.stats['created_missing_labels'] += 1
            
            # Copy image to output
            output_img = output_split / "images" / img_path.name
            shutil.copy2(img_path, output_img)
            
            # Write cleaned label
            output_label = output_split / "labels" / (img_path.stem + '.txt')
            with open(output_label, 'w') as f:
                f.write('\n'.join(fixed_lines))
            
            split_stats['cleaned'] += 1
        
        # Remove orphan labels (labels without images)
        output_labels = output_split / "labels"
        output_images = output_split / "images"
        
        for label_file in output_labels.glob("*.txt"):
            has_image = False
            for ext in IMAGE_EXTENSIONS:
                if (output_images / (label_file.stem + ext)).exists():
                    has_image = True
                    break
            
            if not has_image:
                label_file.unlink()
                self.cleaning_log.append({
                    'action': 'remove_orphan_label',
                    'file': str(label_file),
                    'reason': 'No corresponding image'
                })
                self.stats['removed_orphan_labels'] += 1
        
        return split_stats
    
    def clean_dataset(self) -> Dict:
        """Clean the entire dataset."""
        logger.info("="*60)
        logger.info("Starting Dataset Cleaning")
        logger.info("="*60)
        
        results = {
            'timestamp': datetime.now().isoformat(),
            'source': str(self.source_dir),
            'output': str(self.output_dir),
            'splits': {}
        }
        
        for split in ['train', 'valid', 'test']:
            split_path = self.source_dir / split
            if split_path.exists():
                logger.info(f"\nCleaning {split} split...")
                results['splits'][split] = self.clean_split(split)
        
        # Update overall stats
        for split_data in results['splits'].values():
            self.stats['total_images'] += split_data.get('total', 0)
            self.stats['cleaned_images'] += split_data.get('cleaned', 0)
            self.stats['removed_corrupt'] += split_data.get('removed_corrupt', 0)
            self.stats['removed_duplicates'] += split_data.get('removed_duplicates', 0)
            self.stats['fixed_labels'] += split_data.get('fixed_labels', 0)
        
        results['summary'] = dict(self.stats)
        
        return results

## 3. Run Dataset Cleaning

In [5]:
# Initialize cleaner
cleaner = DatasetCleaner(DATASET_DIR, CLEANED_DIR, num_classes=NUM_CLASSES)

# Run cleaning
cleaning_results = cleaner.clean_dataset()

2026-01-18 21:53:35,882 - INFO - ============================================================
2026-01-18 21:53:35,882 - INFO - Starting Dataset Cleaning
2026-01-18 21:53:35,882 - INFO - ============================================================
2026-01-18 21:53:35,884 - INFO - 
Cleaning train split...
2026-01-18 21:53:35,884 - INFO - Finding duplicates in train...
Computing hashes: 100%|██████████| 2095/2095 [00:09<00:00, 214.61it/s]
2026-01-18 21:53:45,658 - INFO - Found 781 duplicates to remove
2026-01-18 21:53:45,669 - INFO - Processing 2095 images in train...
Cleaning train: 100%|██████████| 2095/2095 [00:07<00:00, 289.80it/s]
2026-01-18 21:53:53,194 - INFO - 
Cleaning valid split...
2026-01-18 21:53:53,196 - INFO - Finding duplicates in valid...
Computing hashes: 100%|██████████| 600/600 [00:02<00:00, 240.99it/s]
2026-01-18 21:53:55,691 - INFO - Found 118 duplicates to remove
2026-01-18 21:53:55,692 - INFO - Processing 600 images in valid...
Cleaning valid: 100%|██████████| 600/

## 4. Cleaning Report

In [6]:
# Print cleaning summary
print("="*70)
print("DATASET CLEANING REPORT")
print("="*70)
print(f"\nSource: {DATASET_DIR}")
print(f"Output: {CLEANED_DIR}")
print(f"Time: {cleaning_results['timestamp']}")

print("\n" + "-"*70)
print("SUMMARY")
print("-"*70)
stats = cleaning_results['summary']
print(f"Total Images Processed:    {stats['total_images']:,}")
print(f"Images Cleaned:            {stats['cleaned_images']:,}")
print(f"Corrupt Images Removed:    {stats['removed_corrupt']:,}")
print(f"Duplicates Removed:        {stats['removed_duplicates']:,}")
print(f"Labels Fixed:              {stats['fixed_labels']:,}")
print(f"Orphan Labels Removed:     {stats['removed_orphan_labels']:,}")
print(f"Missing Labels Created:    {stats['created_missing_labels']:,}")

# Retention rate
if stats['total_images'] > 0:
    retention = stats['cleaned_images'] / stats['total_images'] * 100
    print(f"\nData Retention Rate:       {retention:.2f}%")

DATASET CLEANING REPORT

Source: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\cc-tv-footage-annotation-b8-lcysc-b1-2
Output: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_cleaned
Time: 2026-01-18T21:53:35.884982

----------------------------------------------------------------------
SUMMARY
----------------------------------------------------------------------
Total Images Processed:    2,998
Images Cleaned:            2,052
Corrupt Images Removed:    0
Duplicates Removed:        946
Labels Fixed:              390
Orphan Labels Removed:     0
Missing Labels Created:    0

Data Retention Rate:       68.45%


In [7]:
# Per-split statistics
print("\n" + "-"*70)
print("PER-SPLIT STATISTICS")
print("-"*70)

for split_name, split_data in cleaning_results['splits'].items():
    print(f"\n{split_name.upper()}:")
    print(f"  Total:              {split_data.get('total', 0):,}")
    print(f"  Cleaned:            {split_data.get('cleaned', 0):,}")
    print(f"  Corrupt Removed:    {split_data.get('removed_corrupt', 0):,}")
    print(f"  Duplicates Removed: {split_data.get('removed_duplicates', 0):,}")
    print(f"  Labels Fixed:       {split_data.get('fixed_labels', 0):,}")


----------------------------------------------------------------------
PER-SPLIT STATISTICS
----------------------------------------------------------------------

TRAIN:
  Total:              2,095
  Cleaned:            1,314
  Corrupt Removed:    0
  Duplicates Removed: 781
  Labels Fixed:       249

VALID:
  Total:              600
  Cleaned:            482
  Corrupt Removed:    0
  Duplicates Removed: 118
  Labels Fixed:       97

TEST:
  Total:              303
  Cleaned:            256
  Corrupt Removed:    0
  Duplicates Removed: 47
  Labels Fixed:       44


In [8]:
# Show sample of cleaning actions
print("\n" + "-"*70)
print("SAMPLE CLEANING ACTIONS")
print("-"*70)

action_counts = defaultdict(int)
for log in cleaner.cleaning_log:
    action_counts[log['action']] += 1

print("\nActions Performed:")
for action, count in sorted(action_counts.items(), key=lambda x: -x[1]):
    print(f"  {action}: {count:,}")

print("\nSample Log Entries:")
for log in cleaner.cleaning_log[:10]:
    print(f"  [{log['action']}] {Path(log['file']).name}")
    print(f"    Reason: {log['reason']}")


----------------------------------------------------------------------
SAMPLE CLEANING ACTIONS
----------------------------------------------------------------------

Actions Performed:
  remove_duplicate: 946
  fix_bbox_coordinates: 398
  remove_tiny_bbox: 4

Sample Log Entries:
  [remove_duplicate] fshoplifting_2_11_78_jpg.rf.3709df7afab300d4cdc822ee8896fb4a.jpg
    Reason: Duplicate of fshoplifting_2_11_108_jpg.rf.bc94ee5ebc710c1f46f37beeee5cb781.jpg
  [remove_duplicate] fshoplifting_2_11_84_jpg.rf.c5938925ee21074649e8beee04f32975.jpg
    Reason: Duplicate of fshoplifting_2_11_108_jpg.rf.bc94ee5ebc710c1f46f37beeee5cb781.jpg
  [remove_duplicate] fshoplifting_2_11_90_jpg.rf.b6015169b21dbe25d5b05ed955395c84.jpg
    Reason: Duplicate of fshoplifting_2_11_108_jpg.rf.bc94ee5ebc710c1f46f37beeee5cb781.jpg
  [remove_duplicate] fshoplifting_2_11_948_jpg.rf.a2ca87ac919189ff5fc994073c209eba.jpg
    Reason: Duplicate of fshoplifting_2_11_108_jpg.rf.bc94ee5ebc710c1f46f37beeee5cb781.jpg
  [remove

## 5. Copy data.yaml to Cleaned Directory

In [9]:
# Copy and update data.yaml
source_yaml = DATASET_DIR / "data.yaml"
output_yaml = CLEANED_DIR / "data.yaml"

if source_yaml.exists():
    import yaml
    
    with open(source_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Update paths for cleaned dataset
    data_config['train'] = 'train/images'
    data_config['val'] = 'valid/images'
    data_config['test'] = 'test/images'
    
    with open(output_yaml, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)
    
    print(f"data.yaml copied and updated to: {output_yaml}")
    print("\nContents:")
    print(yaml.dump(data_config, default_flow_style=False))
else:
    print("Warning: No data.yaml found in source directory")

data.yaml copied and updated to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_cleaned\data.yaml

Contents:
names:
- Customer-Bagpack
- Product
- Product-Picked
- Shopping-Cart
- normal
- theft
nc: 6
roboflow:
  license: CC BY 4.0
  project: cc-tv-footage-annotation-b8-lcysc-b1-wutkr
  url: https://universe.roboflow.com/grad-lnwh2/cc-tv-footage-annotation-b8-lcysc-b1-wutkr/dataset/2
  version: 2
  workspace: grad-lnwh2
test: test/images
train: train/images
val: valid/images



## 6. Verify Cleaned Dataset

In [10]:
# Verify the cleaned dataset
print("\n" + "="*70)
print("CLEANED DATASET VERIFICATION")
print("="*70)

for split in ['train', 'valid', 'test']:
    split_path = CLEANED_DIR / split
    if not split_path.exists():
        continue
    
    images_path = split_path / "images"
    labels_path = split_path / "labels"
    
    num_images = len(list(images_path.glob("*.*"))) if images_path.exists() else 0
    num_labels = len(list(labels_path.glob("*.txt"))) if labels_path.exists() else 0
    
    print(f"\n{split.upper()}:")
    print(f"  Images: {num_images:,}")
    print(f"  Labels: {num_labels:,}")
    
    if num_images != num_labels:
        print(f"  WARNING: Image/Label count mismatch!")
    else:
        print(f"  Status: Image-Label pairs verified")


CLEANED DATASET VERIFICATION

TRAIN:
  Images: 1,314
  Labels: 1,314
  Status: Image-Label pairs verified

VALID:
  Images: 482
  Labels: 482
  Status: Image-Label pairs verified

TEST:
  Images: 256
  Labels: 256
  Status: Image-Label pairs verified


In [11]:
# Count annotations in cleaned dataset
print("\n" + "-"*70)
print("CLEANED DATASET CLASS DISTRIBUTION")
print("-"*70)

class_counts = defaultdict(int)
total_annotations = 0

for split in ['train', 'valid', 'test']:
    labels_path = CLEANED_DIR / split / "labels"
    if not labels_path.exists():
        continue
    
    for label_file in labels_path.glob("*.txt"):
        with open(label_file, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    parts = line.split()
                    if len(parts) >= 1:
                        try:
                            class_id = int(parts[0])
                            class_counts[class_id] += 1
                            total_annotations += 1
                        except ValueError:
                            pass

print(f"\nTotal Annotations: {total_annotations:,}")
print("\nClass Distribution:")
for class_id in sorted(class_counts.keys()):
    count = class_counts[class_id]
    pct = count / total_annotations * 100 if total_annotations > 0 else 0
    name = CLASS_NAMES.get(class_id, f'Unknown-{class_id}')
    print(f"  {class_id}: {name:20s} {count:6,} ({pct:5.1f}%)")


----------------------------------------------------------------------
CLEANED DATASET CLASS DISTRIBUTION
----------------------------------------------------------------------

Total Annotations: 7,335

Class Distribution:
  0: Customer-Bagpack        605 (  8.2%)
  1: Product                 793 ( 10.8%)
  2: Product-Picked          778 ( 10.6%)
  3: Shopping-Cart           137 (  1.9%)
  4: normal                4,626 ( 63.1%)
  5: theft                   396 (  5.4%)


## 7. Save Cleaning Report

In [12]:
# Save cleaning report
report_path = OUTPUT_DIR / 'cleaning_report.json'

with open(report_path, 'w') as f:
    json.dump(cleaning_results, f, indent=2)

print(f"Cleaning report saved to: {report_path}")

Cleaning report saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\outputs\cleaning_report.json


In [13]:
# Save detailed cleaning log
log_path = LOG_DIR / 'cleaning_log.json'

with open(log_path, 'w') as f:
    json.dump(cleaner.cleaning_log, f, indent=2)

print(f"Detailed cleaning log saved to: {log_path}")
print(f"Total log entries: {len(cleaner.cleaning_log):,}")

Detailed cleaning log saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\logs\cleaning_log.json
Total log entries: 1,348


## 8. Final Summary

In [14]:
print("\n" + "="*70)
print("DATASET CLEANING COMPLETE")
print("="*70)

print(f"\nCleaned dataset location: {CLEANED_DIR}")
print(f"\nFiles saved:")
print(f"  - Cleaning report: {report_path}")
print(f"  - Detailed log: {log_path}")
print(f"  - data.yaml: {output_yaml}")

print("\n" + "-"*70)
print("NEXT STEPS")
print("-"*70)
print("1. Run 03_dataset_analysis.ipynb for detailed analysis and visualization")
print("2. Run 04_dataset_balancing.ipynb to address class imbalance")
print("="*70)


DATASET CLEANING COMPLETE

Cleaned dataset location: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_cleaned

Files saved:
  - Cleaning report: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\outputs\cleaning_report.json
  - Detailed log: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\logs\cleaning_log.json
  - data.yaml: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_cleaned\data.yaml

----------------------------------------------------------------------
NEXT STEPS
----------------------------------------------------------------------
1. Run 03_dataset_analysis.ipynb for detailed analysis and visualization
2. Run 04_dataset_balancing.ipynb to address class imbalance


---

## Next Steps

After cleaning, proceed to:
1. **03_dataset_analysis.ipynb** - Deep analysis and visualization
2. **04_dataset_balancing.ipynb** - Address class imbalance

---